# MINI_PROJECT_FINAL CODE: bridge crack severity, localization and segmentation

Set the runtime to a T4 GPU first (Runtime > Change runtime type), then run the cells from top to bottom.

Your Drive folder `MyDrive/MINI_PROJECT` needs the project zip and the SDNET2018 zip, named as in Step 3.
If you have put the repository on GitHub, you can set `REPO_URL` in Step 3 and skip the project zip.

Part 1 is the severity pipeline (Steps 1 to 12). Part 2 runs the scripts in `extensions/`. The notebook only
calls the scripts, so everything it does can also be run from a terminal as described in the README.

Severity labels are proxy labels made from measured crack extent. Boxes and masks are weak labels made by the
same measurement code. Scores on them show agreement with that code, not accuracy against human annotation.

# Part 1: severity pipeline

## Step 1: Check the GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("NO GPU -> Runtime > Change runtime type > T4 GPU, then restart and rerun from the top.")

## Step 2: Connect Google Drive

Click Allow when Google asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Unpack the project from Drive

In [ ]:
import os, glob, subprocess

DRIVE_DIR   = "/content/drive/MyDrive/MINI_PROJECT"
PROJECT_ZIP = f"{DRIVE_DIR}/SSL-For-Bridge-Crack-Classification-severity(final).zip"
DATASET_ZIP = f"{DRIVE_DIR}/DATA_Maguire_20180517_ALL (2).zip"
BACKUP      = f"{DRIVE_DIR}/outputs"      # saved results and models go here
REPO_URL    = ""   # optional, e.g. "https://github.com/yourname/SSL-For-Bridge-Crack-Classification.git"

assert os.path.exists(DATASET_ZIP), f"Not found in Drive: {DATASET_ZIP}"
subprocess.run(["rm", "-rf", "/content/project"], check=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/project/repo"], check=True)
else:
    assert os.path.exists(PROJECT_ZIP), f"Not found in Drive: {PROJECT_ZIP}"
    subprocess.run(["unzip", "-q", "-o", PROJECT_ZIP, "-d", "/content/project"], check=True)

# the zip may contain nested folders, so find the one that holds severity/
hits = glob.glob("/content/project/**/severity/make_severity_labels.py", recursive=True)
assert hits, "Could not find severity/make_severity_labels.py in the project"
PROJECT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(PROJECT)
assert os.path.isdir("extensions"), ("This copy of the project has no extensions/ folder. "
                                     "Upload the newest project zip to Drive, or set REPO_URL.")
print("Working folder:", os.getcwd())
print(sorted(os.listdir(".")))

## Step 4: Install the extra libraries

In [ ]:
!pip install -q scikit-image joblib tqdm

## Step 5: Unpack the dataset as Decks / Pavements / Walls

Copying 56,000 images straight from Drive is slow, so the zip is unpacked onto Colab's local disk. This takes a few minutes. The original SDNET2018 uses folders called D, P and W. This cell finds them and builds the layout the code expects.

In [ ]:
!rm -rf /content/data
!unzip -q -o "$DATASET_ZIP" -d /content/data

os.makedirs("/content/archive", exist_ok=True)
for name, cracked_dir in {"Decks": "CD", "Pavements": "CP", "Walls": "CW"}.items():
    hits = [r for r, d, f in os.walk("/content/data")
            if os.path.basename(r) == cracked_dir and "__MACOSX" not in r]
    assert hits, f"Could not find a '{cracked_dir}' folder inside the dataset zip"
    link = f"/content/archive/{name}"
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(os.path.dirname(hits[0]), link)
    print(name, "->", os.path.dirname(hits[0]))

from severity.common import list_dataset
paths, targets, surfaces = list_dataset("/content/archive")
print(f"\n{len(paths)} images in total, {(targets == 0).sum()} cracked, {(targets == 1).sum()} uncracked")
assert len(paths) > 50000, "Expected roughly 56,000 images - check the dataset zip contents"

## Step 6: Self-test of the crack measurement (optional)

In [ ]:
!python severity/selftest.py

## Step 7: Measure cracks and create the proxy severity labels

This runs on the CPU and takes several minutes. Afterwards, look at the contact sheet. Each row is one automatic class. The photo is on top and the detected crack below it. If the marks do not follow real cracks, stop here.

In [ ]:
!python severity/make_severity_labels.py --data_dir /content/archive --workers 2 --spotcheck 12

In [ ]:
# Top of each row = photo, bottom = detected crack. Does the red/yellow overlay follow the real cracks?
from IPython.display import Image, display
display(Image("severity/spotcheck_contact_sheet.png"))

## Step 8: Extract DINOv2 features

This is the slow GPU step. The features are copied to Drive right afterwards so it never has to be repeated.

In [ ]:
!python severity/extract_features.py --data_dir /content/archive --num_workers 2

In [ ]:
import shutil
os.makedirs(BACKUP, exist_ok=True)
for f in ["features_dinov2_vits14.npz", "crack_measurements.csv", "severity_labels.csv", "severity_cutoffs.json"]:
    shutil.copy(f"severity/{f}", BACKUP)
print("Backed up features + labels to", BACKUP)

## Step 9: Train stage 1 (crack / no crack)

In [ ]:
!python severity/train_crack_detector.py

## Step 10: Train and evaluate stage 2 (severity)

In [ ]:
!python severity/train_severity.py

In [ ]:
from IPython.display import Image, display
display(Image("results/severity/severity_confusion_matrix.png"))
display(Image("results/severity/severity_label_efficiency.png"))

## Step 11: Save results and models to Drive

In [ ]:
shutil.copytree("results/severity", f"{BACKUP}/results_severity", dirs_exist_ok=True)
shutil.copytree("severity/models", f"{BACKUP}/models", dirs_exist_ok=True)
print("Saved results and trained models to", BACKUP)

## Step 12: Try both stages on a few test images

In [ ]:
import pandas as pd
df = pd.read_csv("severity/severity_labels.csv")
sample = df[df.split == "test"].sample(5, random_state=0)
args = " ".join(f'"{p}"' for p in sample.path)
!python severity/infer_two_stage.py $args
print("\nAutomatic proxy labels for these images (for comparison):")
print(sample[["path", "severity_name"]].to_string(index=False))

---
# Part 2: extensions

These need Steps 3 to 8 from this session, or the restore cell at the bottom of the notebook. They split the data
by source photo, so their numbers are not comparable with Step 10, which uses the original random split.

In [ ]:
# Check that the labels and features are in place and show the photo-aware split sizes.
!python -c "from extensions.shared import load_severity_split as f; d = f(); print({k: len(d[k]) for k in ('tr', 'va', 'te')})"

## Feature 1: severity with a validation split

Picks the regularisation strength on the validation set, reports the test set once, counts Severe images predicted
as Minor, and prints a table of how many predictions would go to a human reviewer at each confidence threshold.

In [ ]:
!python extensions/severity_validated.py
from IPython.display import Image, display
display(Image("results_extension/severity/severity_confusion_matrix.png"))

Optional: fine-tune the last two DINOv2 blocks and compare with the frozen head on the same test images. This takes longer.

In [ ]:
!python extensions/severity_finetune.py --epochs 5 --unfreeze 2

## Weak labels for localization and segmentation

SDNET2018 has no boxes or masks, so they are generated from the crack measurement code. Look at the preview before
training. If the masks do not follow real cracks, the two features below will not mean anything.

In [ ]:
!python extensions/build_weak_labels.py --archive /content/archive --workers 2
from IPython.display import Image, display
display(Image("results_extension/weak_labels_preview.png"))

## Feature 2: crack localization with YOLO

A box gives the rough position of a crack. It says nothing about width or severity.

In [ ]:
!pip install -q ultralytics
!python extensions/localization_yolo.py --epochs 30
from IPython.display import Image, display
display(Image("results_extension/localization/localization_examples.png"))

## Feature 3: crack segmentation with a U-Net

The threshold is chosen on the validation split. The overlays show the best, typical and worst test cases.

In [ ]:
!python extensions/segmentation_unet.py --epochs 15
from IPython.display import Image, display
display(Image("results_extension/segmentation/training_curves.png"))
display(Image("results_extension/segmentation/segmentation_examples.png"))

## Tables for the paper and saving to Drive

The LaTeX tables are built from the result files, so the paper numbers come from the runs.

In [ ]:
!python extensions/make_paper_tables.py
import shutil, os
out = f"{BACKUP}/extensions"
os.makedirs(out, exist_ok=True)
shutil.copytree("results_extension", f"{out}/results_extension", dirs_exist_ok=True)
shutil.copytree("paper/tables", f"{out}/paper_tables", dirs_exist_ok=True)
if os.path.exists("ext_work/models"):
    shutil.copytree("ext_work/models", f"{out}/models", dirs_exist_ok=True)
print("Saved results, tables and model weights to", out)

---
## If Colab disconnects midway

Colab clears its disk when a session ends. Redo Steps 1 to 5, then run the cell below. It restores the features,
labels and trained models saved in Drive. After that you can continue at Step 9 or in Part 2. The weak-label
dataset is rebuilt by its own cell.

In [ ]:
# Run only after redoing Steps 1 to 5. This skips the slow steps by restoring what was saved in Drive.
import shutil, os
for f in ["features_dinov2_vits14.npz", "crack_measurements.csv", "severity_labels.csv", "severity_cutoffs.json"]:
    if os.path.exists(f"{BACKUP}/{f}"):
        shutil.copy(f"{BACKUP}/{f}", f"severity/{f}")
if os.path.exists(f"{BACKUP}/models"):
    shutil.copytree(f"{BACKUP}/models", "severity/models", dirs_exist_ok=True)
if os.path.exists(f"{BACKUP}/extensions/models"):
    shutil.copytree(f"{BACKUP}/extensions/models", "ext_work/models", dirs_exist_ok=True)
print("Restored. Continue from Step 9, or from Part 2 if you only need the extensions.")